# Heartbeat Classifier – End-to-End Pipeline

**ECG arrhythmia detection and classification using deep learning.**

This notebook orchestrates the complete pipeline: download → preprocess → train → evaluate.
Each stage skips if artifacts already exist (idempotent).

## 1 – Setup & Configuration

In [ ]:
import sys
import subprocess
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from heartbeat_classifier import config

print("Imports successful")

In [ ]:
# Configuration
DATA_ROOT = Path(".").resolve().parent  # Repository root (one level up from notebooks/)
OUTPUT_DIR = DATA_ROOT / "results" / "run_01"  # Training run to use

# Data directories
ORIGINAL_DB = DATA_ROOT / config.DATA_DIRS["original_database"]
ECG_DIR = DATA_ROOT / config.DATA_DIRS["sliced_ecg"]
ANN_DIR = DATA_ROOT / config.DATA_DIRS["sliced_annotations"]

# Key artifact paths
best_model_path = OUTPUT_DIR / "model_best.keras"
history_path = OUTPUT_DIR / "training_history.json"

print(f"Data root:        {DATA_ROOT}")
print(f"Output dir:       {OUTPUT_DIR}")
print(f"Config EPOCHS:    {config.EPOCHS}")
print(f"Config BATCH_SIZE: {config.BATCH_SIZE}")

## 2 – Download Dataset

In [ ]:
# Skip if original_database already exists
if ORIGINAL_DB.exists() and len(list(ORIGINAL_DB.glob("*.dat"))) > 0:
    print(f"✓ Dataset already downloaded ({len(list(ORIGINAL_DB.glob('*.dat')))} records found)")
else:
    print("Downloading MIT-BIH Arrhythmia Database...")
    result = subprocess.run(
        ["hbc-download", "--data-root", str(DATA_ROOT)],
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
    else:
        print("✓ Download complete")

## 3 – Preprocessing

In [ ]:
# Skip if sliced signals already exist
if ECG_DIR.exists() and len(list(ECG_DIR.glob("*/*.csv"))) > 0:
    n_segments = len(list(ECG_DIR.glob("*/*.csv")))
    print(f"✓ Preprocessing already done ({n_segments} ECG segments found)")
else:
    print("Running preprocessing pipeline...")
    result = subprocess.run(
        ["hbc-preprocess", "--data-root", str(DATA_ROOT)],
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
    else:
        print("✓ Preprocessing complete")

## 4 – Training

In [ ]:
# Training runs as a separate process to avoid Jupyter memory limits with TensorFlow+Metal
if best_model_path.exists() and history_path.exists():
    print(f"✓ Trained model found: {best_model_path.name}")
    print(f"✓ Training history found: {history_path.name}")
else:
    print(f"Training model... (up to {config.EPOCHS} epochs, early stopping patience={config.EARLY_STOPPING_PATIENCE})")
    print("This will take ~20 minutes.\n")
    result = subprocess.run(
        ["hbc-train", "--data-root", str(DATA_ROOT), "--output-dir", str(OUTPUT_DIR)],
        stdout=sys.stdout,
        stderr=sys.stdout
    )
    if result.returncode != 0:
        print(f"\nERROR: Training failed (code {result.returncode})")
    else:
        print("\n✓ Training complete")

## 5 – Training History Visualization

In [ ]:
# Load training history
if history_path.exists():
    with open(history_path, "r") as f:
        history = json.load(f)

    # Plot loss and accuracy
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    axes[0].plot(history["loss"], label="Train", alpha=0.7)
    axes[0].plot(history["val_loss"], label="Val", alpha=0.7)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training & Validation Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy
    if "accuracy" in history:
        axes[1].plot(history["accuracy"], label="Train", alpha=0.7)
        axes[1].plot(history["val_accuracy"], label="Val", alpha=0.7)
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")
        axes[1].set_title("Training & Validation Accuracy")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_history_notebook.png", dpi=100, bbox_inches="tight")
    plt.show()

    print(f"✓ Training history plotted ({len(history['loss'])} epochs)")
else:
    print("No training history found")

## 6 – Evaluation

In [ ]:
eval_dir = OUTPUT_DIR / "evaluation"

if (eval_dir / "plots").exists():
    print(f"✓ Evaluation already done")
else:
    print("Running evaluation on test set...")
    result = subprocess.run(
        [
            "hbc-evaluate",
            "--model", str(best_model_path),
            "--data-root", str(DATA_ROOT),
            "--output-dir", str(eval_dir)
        ]
    )
    if result.returncode != 0:
        print(f"ERROR: Evaluation failed with return code {result.returncode}")
    else:
        print("✓ Evaluation complete")

## 7 – Summary

In [ ]:
# Print summary
print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)

artifacts = {
    "Model (final)": OUTPUT_DIR / "model.keras",
    "Model (best)": OUTPUT_DIR / "model_best.keras",
    "Label encoder": OUTPUT_DIR / "label_encoder.pkl",
    "Training history": OUTPUT_DIR / "training_history.json",
}

for name, path in artifacts.items():
    status = "✓" if path.exists() else "✗"
    size = f"({path.stat().st_size / 1e6:.1f} MB)" if path.exists() else ""
    print(f"{status} {name:<20} {path.name:<25} {size}")

if history_path.exists():
    with open(history_path, "r") as f:
        h = json.load(f)
    print(f"\nTraining:")
    print(f"  Epochs completed: {len(h['loss'])}")
    print(f"  Final train loss: {h['loss'][-1]:.4f}")
    print(f"  Final val loss:   {h['val_loss'][-1]:.4f}")
    if "accuracy" in h:
        print(f"  Final train acc:  {h['accuracy'][-1]:.4f}")
        print(f"  Final val acc:    {h['val_accuracy'][-1]:.4f}")

print("\nOutput directory: " + str(OUTPUT_DIR))
print("="*60)